# EDA: `ma_transactions_500.csv`

Exploratory analysis of the project's 500-deal M&A dataset, done to inform the design of the acquirer engine. The notebook only reads the CSV and prints results.

Summary of conclusions: [FINDINGS.md](FINDINGS.md). Section numbers (§) there refer to the headings below.

To run it, install `requirements.txt` in a Python environment, select that environment as the kernel, and run all cells. Every check is deterministic, and the permutation tests use fixed seeds.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Works whether the notebook is run from eda/ or from the repository root.
CSV = next(p for p in [Path("../data/ma_transactions_500.csv"), Path("data/ma_transactions_500.csv")] if p.exists())

# Column meanings. "obs" = confirmed by the brief's field table;
# "guess" = inferred from the column name and values only.
COLUMN_MEANINGS = {
    "transaction_id": ("obs-ish", "Deal ID, format MA-<year>-<seq>"),
    "target_company": ("obs", "Company being acquired (semi-fictional names)"),
    "acquirer": ("obs", "Buyer (real company names)"),
    "sector": ("obs", "Target's healthcare sector"),
    "sub_sector": ("obs", "Target's sub-sector"),
    "deal_year": ("obs", "Year announced"),
    "deal_quarter": ("obs", "Quarter announced"),
    "deal_type": ("obs", "Transaction structure (LBO, Bolt-on, Carve-out, ...)"),
    "geography": ("obs", "Target's US region (or National / Multi-Regional)"),
    "financing_type": ("guess", "How the buyer paid (cash, stock, leverage, earnout)"),
    "deal_size_mm": ("obs", "Enterprise value, USD millions"),
    "target_revenue_mm": ("guess", "Target revenue at deal (LTM?), USD millions"),
    "target_ebitda_mm": ("guess", "Target EBITDA at deal (LTM?), USD millions"),
    "ebitda_margin_pct": ("obs", "Target EBITDA / revenue, percent"),
    "revenue_growth_pct": ("obs", "Target revenue growth, percent (period unstated)"),
    "ev_ebitda_multiple": ("obs", "EV / EBITDA paid"),
    "ev_revenue_multiple": ("obs", "EV / revenue paid"),
    "synergy_pct_of_deal": ("guess", "Expected synergies as % of deal value"),
    "outcome": ("obs", "Deal status: Closed / Withdrawn / Pending / Terminated (+ Rumored)"),
    "strategic_rationale_tags": ("obs", "Pipe-delimited deal-rationale tags"),
    "num_bidders": ("guess", "Number of bidders in the process (competitiveness)"),
    "days_to_close": ("guess", "Days from announcement to close; blank if not closed"),
    "acquirer_type": ("obs", "Strategic vs. Financial Sponsor (PE)"),
    "target_ownership_pre": ("guess", "Target's owner before the deal (Private, PE-Backed, Public, Non-Profit)"),
}

NUMERIC = [
    "deal_size_mm", "target_revenue_mm", "target_ebitda_mm", "ebitda_margin_pct",
    "revenue_growth_pct", "ev_ebitda_multiple", "ev_revenue_multiple",
    "synergy_pct_of_deal", "num_bidders", "days_to_close",
]
CATEGORICAL = [
    "sector", "sub_sector", "deal_quarter", "deal_type", "geography", "financing_type",
    "outcome", "acquirer_type", "target_ownership_pre",
]
SPEC_OUTCOMES = {"Closed", "Withdrawn", "Pending", "Terminated"}

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)


def report_count(label, mask_or_df):
    n = int(mask_or_df.sum()) if isinstance(mask_or_df, pd.Series) else len(mask_or_df)
    print(f"{label}: {n}")
    return n


def permutation_p(values, groups, n_perm=2000, seed=0):
    """Is a per-group mean more spread out than chance? Statistic: variance of group means.
    p = share of label shuffles with at least the observed spread. Small p = groups really differ."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    codes, _ = pd.factorize(groups)
    counts = np.bincount(codes)

    def spread(c):
        return np.var(np.bincount(c, weights=values) / counts)

    observed = spread(codes)
    hits = sum(spread(rng.permutation(codes)) >= observed for _ in range(n_perm))
    return (hits + 1) / (n_perm + 1)

In [2]:
df = pd.read_csv(CSV)
print(f"Loaded {CSV}")

# Masks reused across sections.
closed = df["outcome"] == "Closed"
pe = df["acquirer_type"] == "Financial Sponsor"
hcs = df[df["sector"] == "Healthcare Services"]

Loaded ../data/ma_transactions_500.csv


## 1. Shape and schema

In [3]:
print(f"rows={len(df)}, columns={df.shape[1]}")
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "basis": [COLUMN_MEANINGS.get(c, ("?", ""))[0] for c in df.columns],
    "meaning": [COLUMN_MEANINGS.get(c, ("?", "UNDOCUMENTED"))[1] for c in df.columns],
})
print(schema.to_string())
undocumented = [c for c in df.columns if c not in COLUMN_MEANINGS]
print(f"\nColumns not in meanings map: {undocumented or 'none'}")

rows=500, columns=24
                            dtype    basis                                                                  meaning
transaction_id             object  obs-ish                                          Deal ID, format MA-<year>-<seq>
target_company             object      obs                            Company being acquired (semi-fictional names)
acquirer                   object      obs                                               Buyer (real company names)
sector                     object      obs                                               Target's healthcare sector
sub_sector                 object      obs                                                      Target's sub-sector
deal_year                   int64      obs                                                           Year announced
deal_quarter               object      obs                                                        Quarter announced
deal_type                  object      obs         

## 2. Keys and grain

In [4]:
print(f"transaction_id unique: {df['transaction_id'].is_unique}")
print(f"target_company unique: {df['target_company'].is_unique}")
print(f"exact duplicate rows: {df.duplicated().sum()}")
content_cols = [c for c in df.columns if c != "transaction_id"]
print(f"duplicate rows ignoring transaction_id: {df.duplicated(subset=content_cols).sum()}")
print(f"duplicate (target, acquirer, year): {df.duplicated(['target_company', 'acquirer', 'deal_year']).sum()}")

parts = df["transaction_id"].str.extract(r"^MA-(\d{4})-(\d{4})$")
print(f"IDs not matching MA-YYYY-NNNN: {parts[0].isna().sum()}")
id_year = parts[0].astype(float)
print(f"ID year != deal_year: {(id_year != df['deal_year']).sum()}")
seq = parts[1].astype(int)
print(f"ID sequence: min={seq.min()}, max={seq.max()}, unique={seq.is_unique}, "
      f"contiguous={sorted(seq) == list(range(1, len(df) + 1))}")

transaction_id unique: True
target_company unique: True
exact duplicate rows: 0
duplicate rows ignoring transaction_id: 0
duplicate (target, acquirer, year): 0
IDs not matching MA-YYYY-NNNN: 0
ID year != deal_year: 0
ID sequence: min=1, max=500, unique=True, contiguous=True


### Deals per acquirer (grain check: one acquirer, many deals)

In [5]:
per_acq = df["acquirer"].value_counts()
print(per_acq.describe().to_string())
print(f"acquirers with 1 deal: {(per_acq == 1).sum()}, with >=5 deals: {(per_acq >= 5).sum()}")

count    107.000000
mean       4.672897
std        6.259748
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       25.000000
acquirers with 1 deal: 34, with >=5 deals: 20


## 3. Data quality

### Missing values (non-zero only)

In [6]:
na = df.isna().sum()
print(na[na > 0].to_string() if na.any() else "none")
print("\ndays_to_close missing, by outcome:")
print(pd.crosstab(df["outcome"], df["days_to_close"].isna().rename("days_missing")).to_string())

days_to_close    94

days_to_close missing, by outcome:
days_missing  False  True 
outcome                   
Closed          406      0
Pending           0     21
Rumored           0     10
Terminated        0     14
Withdrawn         0     49


### String hygiene

In [7]:
obj_cols = df.select_dtypes("object").columns
for c in obj_cols:
    s = df[c]
    ws = (s != s.str.strip()).sum()
    dbl = s.str.contains(r"\s{2,}").sum()
    if ws or dbl:
        print(f"{c}: leading/trailing whitespace={ws}, double spaces={dbl}")
print("(no output above = no whitespace issues)")

(no output above = no whitespace issues)


### Case-insensitive collisions in name/category columns (possible label variants)

In [8]:
for c in ["acquirer"] + CATEGORICAL:
    vals = pd.Series(df[c].unique())
    norm = vals.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
    collide = vals[norm.duplicated(keep=False)]
    if len(collide):
        print(f"{c}: {sorted(collide)}")
print("(no output above = no case/punctuation variants)")

(no output above = no case/punctuation variants)


### Acquirer names with parentheticals or suffixes (naming convention check)

In [9]:
odd = sorted(df.loc[df["acquirer"].str.contains(r"[()]|\b(?:Inc|LLC|Corp|LP)\b"), "acquirer"].unique())
print(odd)

['Aspen Dental (ADMI)', 'Cerner (Oracle)']


### Target names with a repeated word (likely generator typo)

In [10]:
rep = df.loc[df["target_company"].str.match(r".*\b(\w+) \1\b"), "target_company"]
print(rep.tolist())

['Integrated Integrated Behavioral']


### Outcome labels outside the brief's list (Closed/Withdrawn/Pending/Terminated)

In [11]:
print(df.loc[~df["outcome"].isin(SPEC_OUTCOMES), "outcome"].value_counts().to_string())

outcome
Rumored    10


### sector vs. sub_sector

In [12]:
same = (df["sector"] == df["sub_sector"]).mean()
print(f"rows where sub_sector == sector: {same:.0%}")

rows where sub_sector == sector: 100%


### Impossible / suspicious numeric values

In [13]:
checks = {
    "deal_size_mm <= 0": df["deal_size_mm"] <= 0,
    "target_revenue_mm <= 0": df["target_revenue_mm"] <= 0,
    "target_ebitda_mm <= 0": df["target_ebitda_mm"] <= 0,
    "ebitda > revenue": df["target_ebitda_mm"] > df["target_revenue_mm"],
    "ev_ebitda < 3 or > 40": ~df["ev_ebitda_multiple"].between(3, 40),
    "ev_revenue < 0.1 or > 20": ~df["ev_revenue_multiple"].between(0.1, 20),
    "margin < 0 or > 60": ~df["ebitda_margin_pct"].between(0, 60),
    "num_bidders < 1": df["num_bidders"] < 1,
    "days_to_close <= 0": df["days_to_close"] <= 0,
    "deal_year outside 2010-2026": ~df["deal_year"].between(2010, 2026),
}
for label, mask in checks.items():
    report_count(label, mask)

deal_size_mm <= 0: 0
target_revenue_mm <= 0: 0
target_ebitda_mm <= 0: 0
ebitda > revenue: 0
ev_ebitda < 3 or > 40: 0
ev_revenue < 0.1 or > 20: 0
margin < 0 or > 60: 0
num_bidders < 1: 0
days_to_close <= 0: 0
deal_year outside 2010-2026: 0


### Outliers by IQR (1.5x) per numeric column

In [14]:
for c in NUMERIC:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    n = ((s < lo) | (s > hi)).sum()
    print(f"{c:22s} n_outliers={n:3d}  fence=[{lo:.1f}, {hi:.1f}]  max={s.max():.1f}")
    if c == "deal_size_mm":
        print(f"{'':22s} high outliers by sector: {df.loc[df[c] > hi, 'sector'].value_counts().to_dict()}")
print("\nLargest 5 deals:")
print(df.nlargest(5, "deal_size_mm")[["transaction_id", "target_company", "acquirer", "sector", "deal_size_mm",
                                       "target_ownership_pre", "deal_type"]].to_string(index=False))

deal_size_mm           n_outliers= 64  fence=[-585.5, 1085.4]  max=14353.7
                       high outliers by sector: {'Health Insurance': 21, 'Pharma/Biotech': 17, 'Medical Devices': 15, 'Healthcare Services': 9, 'Home Health/Hospice': 2}
target_revenue_mm      n_outliers= 67  fence=[-215.9, 402.0]  max=7481.9
target_ebitda_mm       n_outliers= 58  fence=[-32.9, 61.2]  max=766.8
ebitda_margin_pct      n_outliers=  0  fence=[-4.8, 37.3]  max=33.6
revenue_growth_pct     n_outliers=  0  fence=[-3.1, 29.4]  max=27.7
ev_ebitda_multiple     n_outliers=  1  fence=[2.0, 29.5]  max=29.7
ev_revenue_multiple    n_outliers=  0  fence=[-2.3, 8.8]  max=8.4
synergy_pct_of_deal    n_outliers= 20  fence=[-2.4, 17.6]  max=22.0
num_bidders            n_outliers= 59  fence=[0.5, 4.5]  max=5.0
days_to_close          n_outliers=  0  fence=[-134.1, 596.9]  max=420.0

Largest 5 deals:
transaction_id       target_company           acquirer         sector  deal_size_mm target_ownership_pre             dea

### Acquirer with more than one acquirer_type (should be fixed per firm)

In [15]:
types = df.groupby("acquirer")["acquirer_type"].nunique()
print(types[types > 1].to_string() if (types > 1).any() else "none: each acquirer has one type")

none: each acquirer has one type


### Semantic contradictions: deal_type / financing vs. acquirer_type

In [16]:
print(pd.crosstab(df["deal_type"], df["acquirer_type"]).to_string())
print()
print(pd.crosstab(df["financing_type"], df["acquirer_type"]).to_string())
report_count("\nPE buyer + Merger of Equals / SPAC Merger", pe & df["deal_type"].isin(["Merger of Equals", "SPAC Merger"]))
report_count("PE buyer paying in Stock or Cash + Stock", pe & df["financing_type"].isin(["Stock", "Cash + Stock"]))
report_count("Strategic buyer + LBO / MBO / Platform Investment",
             ~pe & df["deal_type"].isin(["Leveraged Buyout", "Management Buyout", "Platform Investment"]))
report_count("Carve-out deal type whose target was Private / Non-Profit",
             (df["deal_type"] == "Carve-out") & df["target_ownership_pre"].isin(["Private", "Non-Profit"]))

acquirer_type          Financial Sponsor  Strategic
deal_type                                          
Bolt-on Acquisition                   35         22
Carve-out                             11         23
Leveraged Buyout                      38         28
Management Buyout                     30         17
Merger of Equals                      28         12
Minority Investment                   23         19
Platform Investment                   36         24
Recapitalization                      30         26
SPAC Merger                           30         20
Strategic Acquisition                 29         19

acquirer_type     Financial Sponsor  Strategic
financing_type                                
All Cash                         44         35
Cash + Stock                     35         33
Earnout                          40         24
Leveraged                        43         23
Mixed                            39         30
Seller Financing                 42         32

12

### Semantic contradictions: acquirer vs. target sector (strategics only)

In [17]:
strat = df[~pe]
sectors_per_strat = strat.groupby("acquirer")["sector"].agg(lambda s: sorted(s.unique()))
multi = sectors_per_strat[sectors_per_strat.str.len() > 1]
print(f"strategic acquirers: {len(sectors_per_strat)}; buying in >1 sector: {len(multi)}")
print(multi.to_string())

strategic acquirers: 92; buying in >1 sector: 2
acquirer
R1 RCM                             [Health IT, Revenue Cycle]
UnitedHealth Group    [Health Insurance, Healthcare Services]


## 4. Distributions

### Numeric summary

In [18]:
print(df[NUMERIC + ["deal_year"]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T.round(2).to_string())

                     count     mean      std     min       5%      25%      50%      75%      95%       max
deal_size_mm         500.0   642.13  1594.26     6.1    15.59    41.10   153.60   458.82  2755.15  14353.70
target_revenue_mm    500.0   276.49   839.41     2.6     5.30    15.82    50.00   170.30  1065.12   7481.90
target_ebitda_mm     500.0    33.90    85.39     0.2     0.80     2.40     7.60    25.92   144.20    766.80
ebitda_margin_pct    500.0    16.45     6.93     1.7     6.20    11.00    15.50    21.52    28.70     33.60
revenue_growth_pct   500.0    13.30     5.45     2.7     5.20     9.07    13.20    17.20    22.61     27.70
ev_ebitda_multiple   500.0    16.20     4.86     7.2     9.10    12.30    15.75    19.20    25.10     29.70
ev_revenue_multiple  500.0     3.32     1.94     0.3     1.08     1.90     2.68     4.67     7.37      8.45
synergy_pct_of_deal  500.0     8.27     4.32     2.0     2.40     5.10     7.90    10.10    17.01     22.00
num_bidders          500.0  

### Categorical: distinct count and top values

In [19]:
for c in CATEGORICAL:
    vc = df[c].value_counts()
    top = ", ".join(f"{k} ({v})" for k, v in vc.head(10).items())
    print(f"{c} [{df[c].nunique()} distinct]: {top}")

sector [10 distinct]: Behavioral Health (70), Home Health/Hospice (62), Physician Groups (60), Health IT (50), Dental (46), Healthcare Services (46), Revenue Cycle (43), Health Insurance (43), Medical Devices (42), Pharma/Biotech (38)
sub_sector [10 distinct]: Behavioral Health (70), Home Health/Hospice (62), Physician Groups (60), Health IT (50), Dental (46), Healthcare Services (46), Revenue Cycle (43), Health Insurance (43), Medical Devices (42), Pharma/Biotech (38)
deal_quarter [4 distinct]: Q2 (142), Q4 (136), Q1 (128), Q3 (94)
deal_type [10 distinct]: Leveraged Buyout (66), Platform Investment (60), Bolt-on Acquisition (57), Recapitalization (56), SPAC Merger (50), Strategic Acquisition (48), Management Buyout (47), Minority Investment (42), Merger of Equals (40), Carve-out (34)
geography [10 distinct]: Midwest (63), Southeast (59), Mountain West (53), Northeast (51), Mid-Atlantic (51), National (51), Great Plains (46), Multi-Regional (44), West Coast (42), Southwest (40)
financi

### Deals by year

In [20]:
print(df["deal_year"].value_counts().sort_index().to_string())

deal_year
2015    16
2016    29
2017    42
2018    48
2019    62
2020    82
2021    79
2022    70
2023    49
2024    23


### Acquirers: distinct count and top 20

In [21]:
print(f"distinct acquirers: {df['acquirer'].nunique()}")
print(df.groupby(["acquirer", "acquirer_type"]).size().sort_values(ascending=False).head(20).to_string())
print("\nDistinct acquirers by type:")
print(df.groupby("acquirer_type")["acquirer"].nunique().to_string())
print("\nDeals by type:")
print(df["acquirer_type"].value_counts().to_string())

distinct acquirers: 107
acquirer                   acquirer_type    
Blackstone                 Financial Sponsor    25
Bain Capital               Financial Sponsor    24
Nordic Capital             Financial Sponsor    24
Vista Equity               Financial Sponsor    24
Welsh Carson               Financial Sponsor    23
Warburg Pincus             Financial Sponsor    23
New Mountain Capital       Financial Sponsor    21
Advent International       Financial Sponsor    18
TPG Capital                Financial Sponsor    18
KKR                        Financial Sponsor    17
Veritas Capital            Financial Sponsor    16
Hellman & Friedman         Financial Sponsor    15
Francisco Partners         Financial Sponsor    14
General Atlantic           Financial Sponsor    14
GTCR                       Financial Sponsor    14
VNS Health                 Strategic             6
CenterWell Home Health     Strategic             6
Universal Health Services  Strategic             5
Amgen        

## 5. Relationships that matter for the target

### Derived fields vs. stated fields

In [22]:
derived = pd.DataFrame({
    "ev_ebitda_calc": df["deal_size_mm"] / df["target_ebitda_mm"],
    "ev_rev_calc": df["deal_size_mm"] / df["target_revenue_mm"],
    "margin_calc": 100 * df["target_ebitda_mm"] / df["target_revenue_mm"],
})
for stated, calc in [("ev_ebitda_multiple", "ev_ebitda_calc"), ("ev_revenue_multiple", "ev_rev_calc"),
                     ("ebitda_margin_pct", "margin_calc")]:
    rel = (derived[calc] - df[stated]).abs() / df[stated]
    print(f"{stated:20s} vs {calc:15s}: median rel diff={rel.median():.2%}, "
          f"within 5%={(rel <= 0.05).mean():.0%}, >20% off={(rel > 0.20).sum()}, max={rel.max():.0%}")
# If any two of (EBITDA, margin, EV/EBITDA) were the "true" pair, the third would reconcile.
print("\nTriangulating the EBITDA-side fields (EV and revenue reconcile, so they are the anchor):")
triangulation = {
    "ev_ebitda_multiple vs ev_revenue / margin": (df["ev_revenue_multiple"] / (df["ebitda_margin_pct"] / 100), df["ev_ebitda_multiple"]),
    "target_ebitda_mm vs revenue * margin": (df["target_revenue_mm"] * df["ebitda_margin_pct"] / 100, df["target_ebitda_mm"]),
    "target_ebitda_mm vs EV / ev_ebitda": (df["deal_size_mm"] / df["ev_ebitda_multiple"], df["target_ebitda_mm"]),
}
for label, (calc, stated) in triangulation.items():
    rel = (calc - stated).abs() / stated
    print(f"  {label:42s}: median rel diff={rel.median():.2%}, within 5%={(rel <= 0.05).mean():.0%}")
# Low-EBITDA rows amplify rounding; show whether disagreement concentrates there.
rel_ebitda = (derived["ev_ebitda_calc"] - df["ev_ebitda_multiple"]).abs() / df["ev_ebitda_multiple"]
print("\nEV/EBITDA rel diff for target_ebitda_mm < 1 vs >= 1:")
print(rel_ebitda.groupby(df["target_ebitda_mm"] < 1).describe()[["count", "50%", "max"]].round(3).to_string())
worst = df.assign(ev_ebitda_calc=derived["ev_ebitda_calc"].round(1), rel=rel_ebitda.round(2)).nlargest(5, "rel")
print("\nWorst EV/EBITDA disagreements:")
print(worst[["transaction_id", "deal_size_mm", "target_ebitda_mm", "ev_ebitda_multiple", "ev_ebitda_calc", "rel"]]
      .to_string(index=False))

ev_ebitda_multiple   vs ev_ebitda_calc : median rel diff=27.76%, within 5%=11%, >20% off=316, max=1055%
ev_revenue_multiple  vs ev_rev_calc    : median rel diff=0.04%, within 5%=100%, >20% off=0, max=1%
ebitda_margin_pct    vs margin_calc    : median rel diff=18.47%, within 5%=15%, >20% off=234, max=352%

Triangulating the EBITDA-side fields (EV and revenue reconcile, so they are the anchor):
  ev_ebitda_multiple vs ev_revenue / margin : median rel diff=27.87%, within 5%=10%
  target_ebitda_mm vs revenue * margin      : median rel diff=18.18%, within 5%=15%
  target_ebitda_mm vs EV / ev_ebitda        : median rel diff=27.76%, within 5%=11%

EV/EBITDA rel diff for target_ebitda_mm < 1 vs >= 1:
                  count    50%     max
target_ebitda_mm                      
False             465.0  0.272  10.555
True               35.0  0.698   7.667

Worst EV/EBITDA disagreements:
transaction_id  deal_size_mm  target_ebitda_mm  ev_ebitda_multiple  ev_ebitda_calc   rel
  MA-2022-0272       

### Target-profile slice: Healthcare Services, by size band (the brief says 46 total, ~12 in $100-400M)

In [23]:
print(f"Healthcare Services deals: {len(hcs)}; closed: {(hcs['outcome'] == 'Closed').sum()}")
bands = pd.cut(df["deal_size_mm"], [0, 50, 100, 400, 1000, 1e9], labels=["<50", "50-100", "100-400", "400-1000", ">1000"])
print(pd.crosstab(df["sector"], bands, margins=True).to_string())
narrow = hcs[hcs["deal_size_mm"].between(100, 400)]
print(f"\nHealthcare Services $100-400M (inclusive): {len(narrow)}; closed: {(narrow['outcome'] == 'Closed').sum()}")
print(f"note: pd.cut bands above are (100, 400]; deals at exactly $100M: {(df['deal_size_mm'] == 100).sum()}")
print("\nHealthcare Services $100-400M deals:")
print(narrow[["transaction_id", "acquirer", "acquirer_type", "deal_size_mm", "ev_ebitda_multiple",
              "ebitda_margin_pct", "geography", "deal_type", "outcome", "target_ownership_pre"]]
      .sort_values("deal_size_mm").to_string(index=False))

Healthcare Services deals: 46; closed: 37
deal_size_mm         <50  50-100  100-400  400-1000  >1000  All
sector                                                         
Behavioral Health     32      11       23         4      0   70
Dental                17       8       14         7      0   46
Health IT             18       4       21         7      0   50
Health Insurance       0       1       13         8     21   43
Healthcare Services    8       5       12        11     10   46
Home Health/Hospice   24       6       21         8      3   62
Medical Devices        1       2       12        12     15   42
Pharma/Biotech         0       5       10         6     17   38
Physician Groups      28       9       17         6      0   60
Revenue Cycle         13      12       10         8      0   43
All                  141      63      153        77     66  500

Healthcare Services $100-400M (inclusive): 12; closed: 10
note: pd.cut bands above are (100, 400]; deals at exactly $100M: 1


### Who buys Healthcare Services, and do they also buy adjacent sectors?

In [24]:
hcs_buyers = hcs["acquirer"].value_counts()
print(f"distinct HCS acquirers: {len(hcs_buyers)}")
print(df[df["acquirer"].isin(hcs_buyers.index)].pivot_table(
    index="acquirer", columns="sector", values="transaction_id", aggfunc="count", fill_value=0)
    .assign(total=lambda t: t.sum(axis=1)).sort_values("total", ascending=False).head(15).to_string())

print("\nHealthcare Services buyers by type:")
print(hcs.groupby("acquirer_type")["acquirer"].agg(deals="count", distinct="nunique").to_string())
print("\nStrategic Healthcare Services buyers:")
print(hcs.loc[hcs["acquirer_type"] == "Strategic", "acquirer"].value_counts().to_string())

distinct HCS acquirers: 26
sector                Behavioral Health  Dental  Health IT  Health Insurance  Healthcare Services  Home Health/Hospice  Medical Devices  Pharma/Biotech  Physician Groups  Revenue Cycle  total
acquirer                                                                                                                                                                                       
Blackstone                            2       2          3                 6                    2                    1                2               1                 2              4     25
Vista Equity                          6       3          0                 1                    4                    3                2               2                 3              0     24
Nordic Capital                        3       2          0                 0                    4                    3                2               0                10              0     24
Bain Capital 

### Sector x acquirer_type (deal counts)

In [25]:
print(pd.crosstab(df["sector"], df["acquirer_type"]).to_string())
print("\nDeals per strategic acquirer:")
print(df[df["acquirer_type"] == "Strategic"].groupby("acquirer").size().describe().round(2).to_string())

acquirer_type        Financial Sponsor  Strategic
sector                                           
Behavioral Health                   41         29
Dental                              30         16
Health IT                           18         32
Health Insurance                    28         15
Healthcare Services                 22         24
Home Health/Hospice                 29         33
Medical Devices                     24         18
Pharma/Biotech                      17         21
Physician Groups                    60          0
Revenue Cycle                       21         22

Deals per strategic acquirer:
count    92.00
mean      2.28
std       1.30
min       1.00
25%       1.00
50%       2.00
75%       3.00
max       6.00


### Sector mix of the top PE sponsors (generalist or specialist?)

In [26]:
top_pe = df[df["acquirer_type"] == "Financial Sponsor"]["acquirer"].value_counts().head(15).index
print(pd.crosstab(df.loc[df["acquirer"].isin(top_pe), "acquirer"], df["sector"]).to_string())
n_sectors = df[df["acquirer_type"] == "Financial Sponsor"].groupby("acquirer")["sector"].nunique()
print(f"\nsectors per sponsor: min={n_sectors.min()}, max={n_sectors.max()}, {n_sectors.sort_values().to_dict()}")

sector                Behavioral Health  Dental  Health IT  Health Insurance  Healthcare Services  Home Health/Hospice  Medical Devices  Pharma/Biotech  Physician Groups  Revenue Cycle
acquirer                                                                                                                                                                                
Advent International                  3       3          1                 3                    1                    0                2               1                 1              3
Bain Capital                          2       3          2                 1                    2                    1                3               1                 7              2
Blackstone                            2       2          3                 6                    2                    1                2               1                 2              4
Francisco Partners                    1       0          1                 

### 'Strong EBITDA margin' threshold: margin percentiles by sector

In [27]:
print(df.groupby("sector")["ebitda_margin_pct"].describe(percentiles=[0.5, 0.75]).round(1)
      [["count", "50%", "75%", "max"]].to_string())

                     count   50%   75%   max
sector                                      
Behavioral Health     70.0  12.8  14.8  17.9
Dental                46.0  16.6  19.8  21.7
Health IT             50.0  22.5  26.0  27.9
Health Insurance      43.0   5.9   9.4  10.9
Healthcare Services   46.0  15.5  16.8  19.5
Home Health/Hospice   62.0  10.9  13.4  15.9
Medical Devices       42.0  25.4  28.1  29.8
Pharma/Biotech        38.0  28.0  29.8  33.6
Physician Groups      60.0  12.0  13.9  17.0
Revenue Cycle         43.0  21.0  23.2  25.3


### Valuation comps: median multiples on closed deals by sector and by acquirer type

In [28]:
print(df[closed].groupby("sector")[["ev_ebitda_multiple", "ev_revenue_multiple"]].median().round(2).to_string())
print()
print(df[closed].groupby("acquirer_type")[["ev_ebitda_multiple", "ev_revenue_multiple"]].median().round(2).to_string())

                     ev_ebitda_multiple  ev_revenue_multiple
sector                                                      
Behavioral Health                  11.8                 2.48
Dental                             14.5                 2.54
Health IT                          21.6                 5.68
Health Insurance                   17.4                 1.27
Healthcare Services                13.6                 2.70
Home Health/Hospice                14.4                 1.97
Medical Devices                    19.8                 5.24
Pharma/Biotech                     25.3                 7.82
Physician Groups                   11.9                 1.72
Revenue Cycle                      17.9                 3.88

                   ev_ebitda_multiple  ev_revenue_multiple
acquirer_type                                             
Financial Sponsor                15.1                 2.48
Strategic                        17.0                 2.98


### Do multiples move with size / margin / growth? (Spearman, closed deals)

In [29]:
print(df.loc[closed, ["ev_ebitda_multiple", "ev_revenue_multiple", "deal_size_mm", "ebitda_margin_pct",
                      "revenue_growth_pct", "num_bidders", "deal_year"]]
      .corr(method="spearman").round(2).to_string())

                     ev_ebitda_multiple  ev_revenue_multiple  deal_size_mm  ebitda_margin_pct  revenue_growth_pct  num_bidders  deal_year
ev_ebitda_multiple                 1.00                 0.59          0.39               0.56                0.21        -0.03      -0.10
ev_revenue_multiple                0.59                 1.00          0.15               0.80                0.30        -0.04      -0.05
deal_size_mm                       0.39                 0.15          1.00               0.13               -0.04         0.02      -0.03
ebitda_margin_pct                  0.56                 0.80          0.13               1.00                0.26        -0.04      -0.08
revenue_growth_pct                 0.21                 0.30         -0.04               0.26                1.00         0.02      -0.04
num_bidders                       -0.03                -0.04          0.02              -0.04                0.02         1.00      -0.03
deal_year                         

### Geography: is it informative for 'regional'?

In [30]:
print(pd.crosstab(df["geography"], df["acquirer_type"]).to_string())

acquirer_type   Financial Sponsor  Strategic
geography                                   
Great Plains                   30         16
Mid-Atlantic                   29         22
Midwest                        38         25
Mountain West                  31         22
Multi-Regional                 25         19
National                       27         24
Northeast                      26         25
Southeast                      36         23
Southwest                      22         18
West Coast                     26         16


### Ownership pre-deal (target is 'private')

In [31]:
print(pd.crosstab(df["target_ownership_pre"], df["acquirer_type"]).to_string())

acquirer_type         Financial Sponsor  Strategic
target_ownership_pre                              
Non-Profit                           44         31
PE-Backed                            91         50
Private                             112         96
Public                               43         33


### Acquirer activity windows (recency signal)

In [32]:
act = df.groupby("acquirer").agg(n=("transaction_id", "count"), first=("deal_year", "min"),
                                 last=("deal_year", "max"), median_size=("deal_size_mm", "median"))
print(act.sort_values("n", ascending=False).head(15).round(1).to_string())

                       n  first  last  median_size
acquirer                                          
Blackstone            25   2015  2024        341.7
Bain Capital          24   2015  2024        102.9
Nordic Capital        24   2016  2024         77.4
Vista Equity          24   2015  2024        153.4
Welsh Carson          23   2017  2023        172.3
Warburg Pincus        23   2016  2024        106.1
New Mountain Capital  21   2015  2024        121.9
Advent International  18   2016  2024        124.0
TPG Capital           18   2016  2023         87.5
KKR                   17   2017  2024        159.7
Veritas Capital       16   2015  2024         37.8
Hellman & Friedman    15   2015  2024        187.3
Francisco Partners    14   2017  2023        192.4
General Atlantic      14   2019  2024        158.2
GTCR                  14   2015  2023        172.8


## 6. Free-text columns

### strategic_rationale_tags: vocabulary

In [33]:
ex = df.assign(tag=df["strategic_rationale_tags"].str.split("|")).explode("tag")
ex["tag"] = ex["tag"].str.strip()
vc = ex["tag"].value_counts()
print(f"distinct tags: {len(vc)}; distinct tag combinations: {df['strategic_rationale_tags'].nunique()}")
print((vc / len(df)).round(2).rename("share_of_deals").to_string())
n_tags = df["strategic_rationale_tags"].str.count(r"\|") + 1
print(f"\ntags per deal: {n_tags.value_counts().sort_index().to_dict()}")
print(f"empty tag tokens: {(ex['tag'] == '').sum()}")

distinct tags: 12; distinct tag combinations: 138
tag
Geographic Expansion     0.90
High Growth              0.41
Margin Improvement       0.31
Platform Build           0.23
Scale                    0.23
Diversification          0.21
Cost Synergies           0.21
Cross-sell               0.21
Market Share             0.19
Vertical Integration     0.18
Technology Capability    0.10
Carve-out Complexity     0.07

tags per deal: {1: 15, 2: 132, 3: 163, 4: 116, 5: 51, 6: 22, 7: 1}
empty tag tokens: 0


### Tag position: is 'Geographic Expansion' always present and where?

In [34]:
first = df["strategic_rationale_tags"].str.split("|").str[0].value_counts()
print("first tag:", first.to_dict())

first tag: {'Geographic Expansion': 304, 'Scale': 114, 'Technology Capability': 50, 'High Growth': 7, 'Diversification': 5, 'Cross-sell': 4, 'Platform Build': 4, 'Vertical Integration': 3, 'Carve-out Complexity': 3, 'Margin Improvement': 3, 'Market Share': 2, 'Cost Synergies': 1}


### Tag vs. deal_type consistency

In [35]:
print(pd.crosstab(ex["tag"], ex["deal_type"]).to_string())
report_count("\n'Carve-out Complexity' tag on non-Carve-out deal",
             df["strategic_rationale_tags"].str.contains("Carve-out Complexity") & (df["deal_type"] != "Carve-out"))
report_count("'Platform Build' tag by Strategic acquirer",
             df["strategic_rationale_tags"].str.contains("Platform Build") & (df["acquirer_type"] == "Strategic"))

deal_type              Bolt-on Acquisition  Carve-out  Leveraged Buyout  Management Buyout  Merger of Equals  Minority Investment  Platform Investment  Recapitalization  SPAC Merger  Strategic Acquisition
tag                                                                                                                                                                                                         
Carve-out Complexity                     0         34                 0                  0                 0                    0                    0                 0            0                      0
Cost Synergies                          21          4                12                  8                10                    5                   17                11            7                     11
Cross-sell                              11          4                15                  8                 5                   10                   15                12           1

46

### Tag share by acquirer type (share of that type's deals)

In [36]:
share = pd.crosstab(ex["tag"], ex["acquirer_type"]) / df["acquirer_type"].value_counts()
print(share.round(2).to_string())

acquirer_type          Financial Sponsor  Strategic
tag                                                
Carve-out Complexity                0.04       0.11
Cost Synergies                      0.23       0.19
Cross-sell                          0.23       0.18
Diversification                     0.19       0.24
Geographic Expansion                0.91       0.89
High Growth                         0.35       0.50
Margin Improvement                  0.25       0.39
Market Share                        0.20       0.18
Platform Build                      0.24       0.22
Scale                               0.21       0.27
Technology Capability               0.06       0.15
Vertical Integration                0.16       0.21


### target_company: name tokens (do names encode sub-specialty?)

In [37]:
last_tokens = df["target_company"].str.split().str[1:].str.join(" ")
print(f"distinct name stems (first word dropped): {last_tokens.nunique()}")
print("Name stems per sector (top 6):")
for s, g in last_tokens.groupby(df["sector"]):
    print(f"  {s}: {g.value_counts().head(6).to_dict()}")

distinct name stems (first word dropped): 110
Name stems per sector (top 6):
  Behavioral Health: {'Mental Health': 14, 'Counseling': 10, 'Addiction Treatment': 10, 'Behavioral Services': 10, 'Integrated Behavioral': 8, 'Behavioral Health': 6}
  Dental: {'Oral Health': 8, 'Orthodontics': 7, 'Dental Care': 7, 'DSO': 6, 'Dental Network': 6, 'Dental Solutions': 4}
  Health IT: {'EHR Solutions': 8, 'Clinical Analytics': 5, 'Revenue Cycle': 4, 'Care Management Tech': 4, 'Care Analytics': 4, 'Health Informatics': 4}
  Health Insurance: {'Indemnity': 7, 'Managed Care': 5, 'Medicare Advantage': 5, 'Health Plan': 4, 'Benefits': 4, 'Health Assurance': 4}
  Healthcare Services: {'Medical Group': 5, 'Care': 5, 'Health Management': 4, 'Care Partners': 4, 'Medical Management': 4, 'Ambulatory Care': 3}
  Home Health/Hospice: {'Personal Care': 8, 'PACE': 7, 'Palliative Care': 7, 'Home Health Partners': 7, 'Home Care': 7, 'Home Infusion': 6}
  Medical Devices: {'Diagnostics': 7, 'MedTech': 6, 'Imaging 

## 7. Acquirer-level and target-slice checks

### A. Closed Healthcare Services deals at $100-400M (the band named in the brief's data note)

In [38]:
comps = df[closed & (df["sector"] == "Healthcare Services") & df["deal_size_mm"].between(100, 400)]
print(f"n={len(comps)}")
print(comps[["ev_ebitda_multiple", "ev_revenue_multiple"]].describe(percentiles=[0.25, 0.5, 0.75])
      .loc[["count", "25%", "50%", "75%", "min", "max"]].round(2).to_string())
print("IDs:", comps["transaction_id"].tolist())

n=10
       ev_ebitda_multiple  ev_revenue_multiple
count               10.00                10.00
25%                 10.30                 2.62
50%                 12.10                 2.90
75%                 14.35                 3.22
min                  9.30                 1.52
max                 16.80                 3.36
IDs: ['MA-2024-0120', 'MA-2021-0195', 'MA-2020-0232', 'MA-2020-0304', 'MA-2020-0306', 'MA-2022-0350', 'MA-2019-0352', 'MA-2021-0407', 'MA-2020-0408', 'MA-2018-0470']


### B. Do strategics pay more than sponsors within a sector? (closed deals; the overall gap may be sector mix)

In [39]:
by = df[closed].groupby(["sector", "acquirer_type"])["ev_ebitda_multiple"].agg(["median", "count"]).unstack()
diff = (by[("median", "Strategic")] - by[("median", "Financial Sponsor")]).rename("strat_minus_pe")
print(pd.concat([by["median"].add_prefix("median_"), by["count"].add_prefix("n_"), diff], axis=1).round(1).to_string())
valid = diff.dropna()
weights = by.loc[valid.index, "count"].sum(axis=1)
print(f"sectors where strategics' median is higher: {(valid > 0).sum()} of {len(valid)}; "
      f"deal-weighted mean within-sector gap: {np.average(valid, weights=weights):+.2f}x")
print("note: Physician Groups has no strategic deals, so it drops out of the within-sector comparison")
band = df[closed & df["deal_size_mm"].between(100, 400)]
by_band = band.groupby(["sector", "acquirer_type"])["ev_ebitda_multiple"].median().unstack()
gap = (by_band["Strategic"] - by_band["Financial Sponsor"]).dropna()
print(f"$100-400M only: strategics higher in {(gap > 0).sum()} of {len(gap)} sectors; median within-sector gap {gap.median():+.2f}x")

                     median_Financial Sponsor  median_Strategic  n_Financial Sponsor  n_Strategic  strat_minus_pe
sector                                                                                                           
Behavioral Health                        12.8              11.0                 32.0         23.0            -1.8
Dental                                   13.5              15.0                 27.0         14.0             1.4
Health IT                                20.8              22.0                 16.0         25.0             1.1
Health Insurance                         17.9              16.8                 22.0         15.0            -1.1
Healthcare Services                      12.2              13.9                 19.0         18.0             1.7
Home Health/Hospice                      14.8              14.3                 23.0         28.0            -0.5
Medical Devices                          19.4              19.8                 18.0    

### C. Do sponsors differ detectably from each other? (15 firms, 14-25 deals each)

Method: for each feature, compare how spread out the per-sponsor means are against 2000 random shuffles of the
acquirer labels. A large p means the sample is too small to detect a difference.

In [40]:
print("Permutation test, 2000 shuffles of acquirer labels, seed 0. A small p is evidence that sponsors differ; "
      "a large p means the sample is too small to detect a difference.")
s = df[pe].copy()
s["log_ev"] = np.log(s["deal_size_mm"])
s["not_closed"] = s["outcome"] != "Closed"
s["regional_target"] = ~s["geography"].isin(["National", "Multi-Regional"])
s["private_target"] = s["target_ownership_pre"].isin(["Private", "PE-Backed"])
s["leveraged"] = s["financing_type"] == "Leveraged"
s["top_quartile_margin_in_sector"] = s["ebitda_margin_pct"] >= s.groupby("sector")["ebitda_margin_pct"].transform(lambda x: x.quantile(0.75))
for dt in ["Platform Investment", "Bolt-on Acquisition", "Leveraged Buyout"]:
    s[f"type={dt}"] = s["deal_type"] == dt
for tag in ["High Growth", "Margin Improvement", "Scale", "Cost Synergies", "Cross-sell", "Vertical Integration"]:
    s[f"tag={tag}"] = s["strategic_rationale_tags"].str.contains(tag, regex=False)
tests = ["log_ev", "num_bidders", "not_closed", "regional_target", "private_target", "leveraged",
         "top_quartile_margin_in_sector", "ev_ebitda_multiple", "synergy_pct_of_deal"] + \
        [c for c in s.columns if c.startswith(("type=", "tag="))]
rows = [(t, permutation_p(s[t], s["acquirer"])) for t in tests]
rows.append(("sector (Physician Groups share)", permutation_p(s["sector"] == "Physician Groups", s["acquirer"])))
print(pd.DataFrame(rows, columns=["feature", "p_value"]).sort_values("p_value").round(3).to_string(index=False))
print(f"note: {len(rows)} tests, so ~{0.05 * len(rows):.0f} p < 0.05 is expected by chance alone")

print("\nCategorical mix per sponsor (chi-square statistic vs. 2000 label shuffles, seed 0):")
rng = np.random.default_rng(0)
for col in ["sector", "deal_type", "geography", "financing_type", "target_ownership_pre"]:
    def chi2(labels):
        t = pd.crosstab(labels, s[col].to_numpy()).to_numpy(float)
        e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / t.sum()
        return ((t - e) ** 2 / e).sum()
    acq = s["acquirer"].to_numpy()
    obs = chi2(acq)
    null = np.array([chi2(rng.permutation(acq)) for _ in range(2000)])
    print(f"  {col:<22} p={(np.sum(null >= obs) + 1) / 2001:.3f}")

Permutation test, 2000 shuffles of acquirer labels, seed 0. A small p is evidence that sponsors differ; a large p means the sample is too small to detect a difference.


                        feature  p_value
       type=Bolt-on Acquisition    0.016
                         log_ev    0.093
          type=Leveraged Buyout    0.225
                regional_target    0.245
         tag=Margin Improvement    0.301
                      tag=Scale    0.329
             ev_ebitda_multiple    0.404
       type=Platform Investment    0.420
                tag=High Growth    0.469
sector (Physician Groups share)    0.482
                      leveraged    0.498
                 private_target    0.529
  top_quartile_margin_in_sector    0.535
                     not_closed    0.586
                    num_bidders    0.669
       tag=Vertical Integration    0.761
            synergy_pct_of_deal    0.820
                 tag=Cross-sell    0.929
             tag=Cost Synergies    0.941
note: 19 tests, so ~1 p < 0.05 is expected by chance alone

Categorical mix per sponsor (chi-square statistic vs. 2000 label shuffles, seed 0):


  sector                 p=0.425


  deal_type              p=0.191


  geography              p=0.208


  financing_type         p=0.621


  target_ownership_pre   p=0.156


### D. Do strategics concentrate in one region? (vs. shuffled geography; strategics with >=2 deals in a named region)

In [41]:
st_df = df[~pe & ~df["geography"].isin(["National", "Multi-Regional"])]
multi = st_df.groupby("acquirer").filter(lambda g: len(g) >= 2)
observed = (multi.groupby("acquirer")["geography"].nunique() == 1).mean()
rng = np.random.default_rng(0)
null = [(multi.assign(geography=rng.permutation(multi["geography"].to_numpy()))
         .groupby("acquirer")["geography"].nunique() == 1).mean() for _ in range(2000)]
print(f"strategics: {multi['acquirer'].nunique()}; share with all regional deals in one region: {observed:.0%} "
      f"(shuffled mean {np.mean(null):.0%}, p={(np.sum(np.array(null) >= observed) + 1) / 2001:.3f})")

strategics: 47; share with all regional deals in one region: 6% (shuffled mean 6%, p=0.535)


### E. Acquirers whose largest deal is below $200M

In [42]:
mx = df.groupby(["acquirer", "acquirer_type"])["deal_size_mm"].max().reset_index()
print(mx.assign(below=mx["deal_size_mm"] < 200).groupby("acquirer_type")["below"].agg(["sum", "count"]).to_string())

                   sum  count
acquirer_type                
Financial Sponsor    0     15
Strategic           26     92


### F. Is 2024 a partial year? (deals by quarter, last 3 years)

In [43]:
print(pd.crosstab(df["deal_year"], df["deal_quarter"]).loc[2022:].to_string())

deal_quarter  Q1  Q2  Q3  Q4
deal_year                   
2022          18  21  13  18
2023          13  17   8  11
2024           7   6   7   3


### G. Closed $100-400M deals since 2020, per acquirer (Healthcare Services vs. any sector)

In [44]:
recent = df[closed & df["deal_size_mm"].between(100, 400) & (df["deal_year"] >= 2020)]
per_acq = pd.DataFrame({
    "hcs": recent[recent["sector"] == "Healthcare Services"].groupby("acquirer").size(),
    "any_sector": recent.groupby("acquirer").size(),
}).fillna(0).astype(int)
per_acq["type"] = df.groupby("acquirer")["acquirer_type"].first().reindex(per_acq.index)
print(f"acquirers with >=1 in HCS: {(per_acq['hcs'] >= 1).sum()}; with >=1 in any sector: {len(per_acq)}")
print(per_acq.groupby("type")[["hcs", "any_sector"]].describe().loc[:, (slice(None), ["count", "mean", "max"])].round(2).to_string())
print(per_acq.sort_values(["hcs", "any_sector"], ascending=False).head(15).to_string())

acquirers with >=1 in HCS: 7; with >=1 in any sector: 41
                    hcs            any_sector           
                  count  mean  max      count  mean  max
type                                                    
Financial Sponsor  15.0  0.40  2.0       15.0  3.73  7.0
Strategic          26.0  0.08  1.0       26.0  1.15  3.0
                          hcs  any_sector               type
acquirer                                                    
Blackstone                  2           3  Financial Sponsor
Advent International        1           6  Financial Sponsor
TPG Capital                 1           5  Financial Sponsor
Bain Capital                1           4  Financial Sponsor
Francisco Partners          1           4  Financial Sponsor
Atrium Health               1           1          Strategic
Bon Secours Mercy Health    1           1          Strategic
Nordic Capital              0           7  Financial Sponsor
Warburg Pincus              0           5  Finan